# Build a Reasoning Chatbot w/ DeepSeek
This will be the first notebook built on top of the LangChain v1.0 pre-release.

This notebook is supposed to be the masterdoc for creating a chatbot that has access to memory, tools, and works as an agent, which means it will work in loops to get the desired outcome.

We will start super basic and just make a robot respond to us, move on to creating an agent with tools, short-term memory, and give it a simpler, more friendly companion to "speak" for it. Models can choose between calling tools, generating structured output, or both.

## Setup

Let's start by installing and importing the necessary libraries.

In [1]:
%pip install --pre -Uq langchain langchain-community langchain-deepseek langgraph langsmith
%pip install -Uq python-dotenv google-search-results

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Environment Variables
Let's make sure we securely store and load our API key(s). We'll use `load_dotenv()` to load our DeepSeek API key, and our LangSmith API key if we choose to do so.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

True

### Model Initialization

We'll be using `ChatDeepSeek`, which uses [`BaseChatOpenAI`](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.BaseChatOpenAI.html#langchain_openai.chat_models.base.BaseChatOpenAI "LangChain API Reference") under the hood to initialize a chatbot using the appropriate `api_base`, which we would otherwise have to set if we chose to use [`init_chat_model`](https://python.langchain.com/api_reference/langchain/chat_models/langchain.chat_models.base.init_chat_model.html).

Next we'll import the necessary modules to do a quick test that we can invoke our chatbot.

In [3]:
from langchain_deepseek import ChatDeepSeek

# Initialize our model
model = ChatDeepSeek(
    temperature=0.7,  # Higher temp for creative responses
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    model="deepseek-chat", # "chat" model for a simple task
    max_tokens=128, # Keep spending use to a minimum
    streaming=True
)

## Generating Responses

In [4]:
# Test model invocation
response = model.invoke("briefly explain why trans women are 'real women.'")
print(response)

content="Of course. Here is a brief explanation:\n\nTrans women are real women because **womanhood is defined by one's internal, deeply-held sense of self (gender identity), not by the sex assigned at birth.**\n\nBeing a woman is a lived experience of identity, social role, and often, a journey of aligning one's outward presentation with that inner truth. While chromosomes and anatomy are one way to categorize people, they are not the sole definition of a complex human experience.\n\nThe medical and scientific consensus, including organizations like the American Medical Association and the American Psychological Association, affirms that transgender identities are valid. Therefore, a trans woman who identifies" additional_kwargs={} response_metadata={'finish_reason': 'length', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_08f168e49b_prod0820_fp8_kvcache'} id='run--ebaf7420-d076-48c9-a8fd-4c6d284039a6-0' usage_metadata={'input_tokens': 15, 'output_tokens': 127, 'total_tokens'

Awesome! It works.

![CHAD](../../assets/CHAD.png)

### Streaming

Now let's try streaming, since it's so engaging and will helpful to learn now in case of future chatbot projects.

We'll make a very simple change from `invoke` to `stream`, and it should just... work. Note that `invoke` may automatically stream content if available. Not quite sure how it works, just know that LangChain made that [claim](https://docs.langchain.com/oss/python/langchain/models#how-it-works).

In [5]:
full = None  # None | AIMessageChunk
for chunk in model.stream("i used to be an adventurer like you, then i took an arrow in the knee"):
    full = chunk if full is None else full + chunk
    print(full.text)

<bound method BaseMessage.text of AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='run--1bebb3ea-2dab-4d0d-8936-6c5d916d4e38')>
<bound method BaseMessage.text of AIMessageChunk(content='Of', additional_kwargs={}, response_metadata={}, id='run--1bebb3ea-2dab-4d0d-8936-6c5d916d4e38')>
<bound method BaseMessage.text of AIMessageChunk(content='Of course', additional_kwargs={}, response_metadata={}, id='run--1bebb3ea-2dab-4d0d-8936-6c5d916d4e38')>
<bound method BaseMessage.text of AIMessageChunk(content='Of course!', additional_kwargs={}, response_metadata={}, id='run--1bebb3ea-2dab-4d0d-8936-6c5d916d4e38')>
<bound method BaseMessage.text of AIMessageChunk(content='Of course! That', additional_kwargs={}, response_metadata={}, id='run--1bebb3ea-2dab-4d0d-8936-6c5d916d4e38')>
<bound method BaseMessage.text of AIMessageChunk(content='Of course! That line', additional_kwargs={}, response_metadata={}, id='run--1bebb3ea-2dab-4d0d-8936-6c5d916d4e38')>
<bound method BaseMe

Nice. So we used 22 input tokens, and used the maximum allotted tokens, 128, bringing our total to 150.

That's like 0.00015 cents!

Now let's try creating an Agent so the chatbot can use tools, like weather checking, or a calculator.

### Agents

"Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions." [Source](https://docs.langchain.com/oss/python/langchain/agents)

When we use `create_agent`, we'll be initializing a ReAct Agent. 

ReAct frames an agent’s behavior as an interleaving of thought -> action -> observation steps, where the model writes out its reasoning, picks a tool, sees the tool’s result, and then repeats. ReAct reduces hallucinations and makes the decision process auditable: the agent can form hypotheses (thought), test them with tools (action), and update its plan based on feedback (observation). A ReAct loop runs until a stop condition - i.e. when the model emits a final answer or a max-iterations limit is reached. [Source](https://docs.langchain.com/oss/python/langchain/agents#overview)

We'll need to initialize the Reasoner model from DeepSeek, initialize the agent, assign it a tool, and then test it's reasoning by asking it to answer a simple question that requires weather information.

But before we do any of that, we need to create the apppriate tools. I'll be creating a couple functions, two of them being tools, and one utility function to format the output of the weather tool.

In [6]:
from serpapi import GoogleSearch

# First, create the output formatting function
def format_weather_data(text_blocks):
    """Format the weather data from text_blocks into a readable string"""
    formatted_text = ""
    
    for block in text_blocks:
        if block["type"] == "paragraph":
            formatted_text += f"{block['snippet']}\n\n"
        elif block["type"] == "heading":
            formatted_text += f"--- {block['snippet']} ---\n"
        elif block["type"] == "list" and "list" in block:
            for item in block["list"]:
                formatted_text += f"• {item['snippet']}\n"
            formatted_text += "\n"
    
    return formatted_text.strip()

# Then, import the tool module and create the weather tool
from langchain_core.tools import tool

# Define the tool using the langchain decorator
@tool
def get_weather(location: str) -> str:
    """Get current weather information for a specific location.
    Example: "portland oregon", "new york", "london uk"
    """
    try:
        params = {
            "engine": "google_ai_mode",
            "q": f"{location} weather",
            "api_key": os.environ["SERPAPI_KEY"],
            "location": "Portland, OR"
        }
        
        search = GoogleSearch(params)
        results = search.get_dict()
        
        if "text_blocks" in results and len(results["text_blocks"]) > 0:
            # Extract and format weather information from the text blocks
            weather_info = format_weather_data(results["text_blocks"])
            return f"Weather in {location}:\n{weather_info}"
        else:
            return f"Could not find weather information for {location}"
            
    except Exception as e:
        return f"Error fetching weather data: {str(e)}"

**Sample output:**
Weather in Portland Oregon:
Current conditions (as of 11:00 PM CDT, September 12, 2025):

• Temperature: 62°F
• Humidity: 81%
• Wind: 3 mph from the north
• Chance of rain: 0%

Short-term forecast:

• Today (Friday): Partly cloudy with a high of 71°F and a low of 55°F. There is a 10% chance of rain during the day.
• Saturday: Partly cloudy with a high of 78°F. A 75% chance of light rain is expected at night, with a low of 59°F.
• Sunday: Light rain is likely with a high of 68°F. The low will be 53°F and there is a 70% chance of rain during the day.

Looking ahead:

• Rain showers are expected to continue into early next week before skies clear up.
• Temperatures will climb into the high 80s by the middle of next week.
• Warm, dry summers are typical for Portland, with rain returning more commonly in mid-October.

--- 

Directly testing the tool would throw an error but we can quickly test the above login in a few short steps, if necessary.

In [7]:
params = {
  "engine": "google_ai_mode",
  "q": "portland oregon weather",
  "api_key": os.environ["SERPAPI_KEY"]
}

search = GoogleSearch(params)
results = search.get_dict()
text_blocks = results["text_blocks"]
print(text_blocks)

[{'type': 'paragraph', 'snippet': 'As of Saturday, September 13, 2025, at 5:00 PM CDT, the weather in Portland, Oregon is 77°F with mostly sunny conditions expected later this afternoon.'}, {'type': 'paragraph', 'snippet': 'Current conditions:', 'snippet_highlighted_words': ['Current conditions']}, {'type': 'list', 'list': [{'snippet': 'Temperature: 77°F'}, {'snippet': 'Wind: 4 mph from the southwest'}, {'snippet': 'Humidity: 45%'}, {'snippet': 'Precipitation: 0% chance of rain'}]}, {'type': 'paragraph', 'snippet': 'Weekend weather', 'snippet_highlighted_words': ['Weekend weather']}, {'type': 'list', 'list': [{'snippet': 'Saturday Night: Clouds will increase, and there is a 55% chance of light rain after 3 a.m..'}, {'snippet': 'Sunday: Widespread light rainfall is likely in the morning, with a high of 68°F and a 45% chance of rain. Conditions will become partly cloudy by the evening.'}]}, {'type': 'paragraph', 'snippet': 'Looking ahead', 'snippet_highlighted_words': ['Looking ahead']},

Next, we need to create the calculator.

In [8]:
import math

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. You can use basic operators (+, -, *, /, ^) and functions like sqrt, sin, cos, etc.
    Example expressions: 
    - "2 + 3 * 4" 
    - "sqrt(16)" 
    - "sin(30) + cos(60)"
    """
    try:
        # Replace ^ with ** for exponentiation
        expression = expression.replace('^', '**')
        
        # Add math functions to the evaluation context
        safe_dict = {
            'abs': abs, 'round': round, 'min': min, 'max': max,
            'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 
            'tan': math.tan, 'log': math.log, 'log10': math.log10,
            'pi': math.pi, 'e': math.e
        }
        
        # Evaluate the expression safely
        result = eval(expression, {"__builtins__": None}, safe_dict)
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

Now we can initialize our reasoner model and create the agent itself. However, it's important to note that including tools automatically [passes the message to the chat model instead.](https://api-docs.deepseek.com/quick_start/pricing "See the (1) footnote under the model's tables.")

In [9]:
# Initialize our model
reasoner_model = ChatDeepSeek(
    temperature=0.2,  # Lower temp for precision
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    model="deepseek-reasoner", # Reasoner!!
    max_tokens=None, # Let it think and work
    streaming=True
)

Since DeepSeek-Reasoner can't use tools, we'll quickly invoke a test to the reasoner and prove it works.

In [10]:
test_reasoner = reasoner_model.invoke("meowww!!! tell me a SUPER CUTE YURI STORY")

print(test_reasoner)
print(f"Story created by '{test_reasoner.response_metadata["model_name"]}'")

content='Of course! Get ready for a fluff overload! Here is a super cute yuri story for you. ❤️\n\n***\n\n### The Cat Café and the Girl with Sunflower Eyes\n\nThe bell above the door of “The Purrfect Cup” chimed, but Elara hardly heard it. She was buried under a mountain of textbooks, her brow furrowed in concentration. Finals were looming, and the cozy cat café was her only sanctuary away from her noisy dorm.\n\nWell, almost her only sanctuary.\n\nThe other part of it was the café’s owner, a girl named Chloe. Chloe was a human sunbeam. She had a laugh that sounded like wind chimes and a habit of wearing oversized sweaters that always smelled like cinnamon and coffee grounds. And her eyes… Elara thought they were the exact warm, golden-brown of a sunflower’s center.\n\nElara came for the quiet and the free Wi-Fi, but she stayed for the five minutes each day when Chloe would bring her a fresh latte—always with a perfect leaf design in the foam—and ask about her day.\n\nToday, however, E

In [11]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

# Create agent with thinker model and calculator tool
agent = create_agent(
    model=reasoner_model,
    tools=[get_weather, calculate],
    # We will leverage SystemMessage to better direct bot behavior
    prompt=SystemMessage(content="You are an AI personal assistant. Use your tools as necessary to complete the user's request(s). You have a calculator, a web searching tool, and a weather searching tool. Be concise and logical.")
)

Now let's test the agent's ability to choose which tool to use and when, and also to prove that the DeepSeek-Chat model is called by reviewing the output.

In [ ]:
# Invoke the agent
agent_test = agent.invoke({"messages": "check the current weather for portland oregon, tell me what that is, and what the square root of that temperature is."})

from pprint import pprint
pprint(agent_test)

{'messages': [HumanMessage(content='check the current weather for portland oregon, tell me what that is, and what the square root of that temperature is.', additional_kwargs={}, response_metadata={}, id='dbc4bb61-d5e2-4584-ba43-95798c8e5166'),
              AIMessage(content="I'll check the current weather for Portland, Oregon and then calculate the square root of the temperature.", additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_00_GzDIINKKDhObwYpxfGYHriL8', 'function': {'arguments': '{"location": "portland oregon"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_08f168e49b_prod0820_fp8_kvcache'}, id='run--9f36d687-4580-49ab-8155-7a3ba31d3ff5-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'portland oregon'}, 'id': 'call_00_GzDIINKKDhObwYpxfGYHriL8', 'type': 'tool_call'}], usage_metadata={'input_tokens': 328, 'output_tokens': 37, 'total_tokens': 365, 'inp

Awesome. It works!!! And yup, the `model_name` is "deepseek-chat".

### Short-term Memory

We'll have to re-initialize our agent, this time, with memory. Let's import and use [`InMemorySaver()`](https://docs.langchain.com/oss/python/langchain/short-term-memory#usage) as our checkpointer before assigning a `thread_id` as a configurable.

Since we're using memory, the agent will need to know "roles" of messages, which we'll implement via `SystemMessage`, etc. as you'll see in the code later. When tools are called, they'll be marked in memory automatically as `ToolMessage`s.

Let's also quickly create a web-search tool and a clock tool for extra fun and the sake of expanding the bot's capabilities.

In [13]:
@tool
def web_search(query: str) -> str:
    """Search the web using Google. Pass in the query you want to search. You can use anything that you would use in a regular Google search. e.g. inurl:, site:, intitle:. 
    """
    try:
        params = {
            "engine": "google_ai_mode",
            "q": f"{query}",
            "api_key": os.environ["SERPAPI_KEY"],
            "location": "Portland, OR"
        }
        
        search = GoogleSearch(params)
        results = search.get_dict()
        
        if "text_blocks" in results and len(results["text_blocks"]) > 0:
            # Extract and format weather information from the text blocks
            output = results["text_blocks"]
            return f"{query} results:\n{output}"
        else:
            return f"Could not find search Google for {query}"
            
    except Exception as e:
        return f"Error fetching weather data: {str(e)}"

from datetime import datetime

@tool
def clock():
    """Get the datetime. Returns datetime.now().strftime("%Y-%m-%d %I:%M %p")"""    
    return datetime.now().strftime("%Y-%m-%d %I:%M %p")

Let's be sure to re-initialize our agent with more capabilities, including memory, and the new `web_search` and `clock` tools.

In [14]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

# Create agent with thinker model and calculator tool
agent = create_agent(
    reasoner_model,
    tools=[get_weather, calculate, web_search, clock],
    # We will leverage SystemMessage to better direct bot behavior
    prompt=SystemMessage(content="You are a reasoning agent. Analyze the user's message and provide step-by-step reasoning. Use the calculator tool when mathematical calculations are needed. You may also check the weather by directly passing the target location into the tool. You have a web search tool that returns an AI summary of Google's results to any search query. Be concise and logical."),
    checkpointer=checkpointer
)

Now we get to test out tools and memory simultaneously. We need to import and leverage `HumanMessage`, as mentioned earlier, to let the bot know who's "speaking."

In [15]:
from langchain_core.messages import HumanMessage

# Invoke agent with thread_id for memory persistence and proper message classes
agent_test = agent.invoke(
    {
        "messages": [
            HumanMessage(content="check the current weather for portland oregon, tell me what that is, and what the square root of that temperature is.")
        ]
    },
    {"configurable": {"thread_id": "1"}}
)

pprint(agent_test)

{'messages': [HumanMessage(content='check the current weather for portland oregon, tell me what that is, and what the square root of that temperature is.', additional_kwargs={}, response_metadata={}, id='da808aad-0efd-400f-8649-b7433c91a303'),
              AIMessage(content="I'll help you check the weather in Portland, Oregon and then calculate the square root of the temperature.", additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_00_dVm9PMAxgtK4m8UCC0mJob6T', 'function': {'arguments': '{"location": "portland oregon"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_08f168e49b_prod0820_fp8_kvcache'}, id='run--9ecfa3be-5f23-46fb-a5ce-78836ff3a734-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'portland oregon'}, 'id': 'call_00_dVm9PMAxgtK4m8UCC0mJob6T', 'type': 'tool_call'}], usage_metadata={'input_tokens': 472, 'output_tokens': 38, 'total_tokens': 510, 'inp

Awesome! So the mf bot can correctly choose to use `get_weather` over the `web_search` function, as seen in line one under `function`. Then, in the final `AIMessage`, we see the bot finish up having used two tools to accomplish it's goal, and give itself the "stop" command.

Now we need to make sure the memory is working by asking a follow-up question.

In [16]:
# Subsequent calls with same thread_id will maintain memory
agent_test2 = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Now what was the temperature you found earlier?")
        ]
    },
    {"configurable": {"thread_id": "1"}}  # Same thread_id maintains conversation
)

pprint(agent_test2)

{'messages': [HumanMessage(content='check the current weather for portland oregon, tell me what that is, and what the square root of that temperature is.', additional_kwargs={}, response_metadata={}, id='da808aad-0efd-400f-8649-b7433c91a303'),
              AIMessage(content="I'll help you check the weather in Portland, Oregon and then calculate the square root of the temperature.", additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_00_dVm9PMAxgtK4m8UCC0mJob6T', 'function': {'arguments': '{"location": "portland oregon"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_08f168e49b_prod0820_fp8_kvcache'}, id='run--9ecfa3be-5f23-46fb-a5ce-78836ff3a734-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'portland oregon'}, 'id': 'call_00_dVm9PMAxgtK4m8UCC0mJob6T', 'type': 'tool_call'}], usage_metadata={'input_tokens': 472, 'output_tokens': 38, 'total_tokens': 510, 'inp

In [17]:
# Test the web_search tool
agent_test3 = agent.invoke(
    {
        "messages": [
            HumanMessage(content="will you look up Portland's news?<3")
        ]
    },
    {"configurable": {"thread_id": "1"}}
)

pprint(agent_test3)

{'messages': [HumanMessage(content='check the current weather for portland oregon, tell me what that is, and what the square root of that temperature is.', additional_kwargs={}, response_metadata={}, id='da808aad-0efd-400f-8649-b7433c91a303'),
              AIMessage(content="I'll help you check the weather in Portland, Oregon and then calculate the square root of the temperature.", additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_00_dVm9PMAxgtK4m8UCC0mJob6T', 'function': {'arguments': '{"location": "portland oregon"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_08f168e49b_prod0820_fp8_kvcache'}, id='run--9ecfa3be-5f23-46fb-a5ce-78836ff3a734-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'portland oregon'}, 'id': 'call_00_dVm9PMAxgtK4m8UCC0mJob6T', 'type': 'tool_call'}], usage_metadata={'input_tokens': 472, 'output_tokens': 38, 'total_tokens': 510, 'inp

The agent chose the `web_search` tool as intended. Nice. Let's quickly test the `clock` tool and then wrap up.

In [ ]:
# Test the clock tool
agent_test4 = agent.invoke(
    {
        "messages": [
            HumanMessage(content="what's the current time?")
        ]
    },
    {"configurable": {"thread_id": "1"}}
)

pprint(agent_test4)

{'messages': [HumanMessage(content='check the current weather for portland oregon, tell me what that is, and what the square root of that temperature is.', additional_kwargs={}, response_metadata={}, id='da808aad-0efd-400f-8649-b7433c91a303'),
              AIMessage(content="I'll help you check the weather in Portland, Oregon and then calculate the square root of the temperature.", additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_00_dVm9PMAxgtK4m8UCC0mJob6T', 'function': {'arguments': '{"location": "portland oregon"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_08f168e49b_prod0820_fp8_kvcache'}, id='run--9ecfa3be-5f23-46fb-a5ce-78836ff3a734-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'portland oregon'}, 'id': 'call_00_dVm9PMAxgtK4m8UCC0mJob6T', 'type': 'tool_call'}], usage_metadata={'input_tokens': 472, 'output_tokens': 38, 'total_tokens': 510, 'inp

## Wrapping Up
Great job so far! We made a lot of progress in a short amount of time. Try customizing the agent(s) if you haven't already, and then start thinking about what tools you'd like to give your AI. You could hook up an Agent to Tools that allow it to interact with the real world, if you'd like!